# RoadScan - Dataset Download
Download and prepare RDD2022 dataset.

In [ ]:
import os
from pathlib import Path
DATA_DIR = Path('/kaggle/working/roadscan')
RAW_DIR = DATA_DIR / 'raw'
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
print('Downloading RDD2022...')
!wget -O {DATA_DIR}/rdd2022.zip https://figshare.com/ndownloader/articles/21431547/versions/1
print('Done!')

In [ ]:
print('Extracting...')
!unzip -q {DATA_DIR}/rdd2022.zip -d {RAW_DIR}
!ls {RAW_DIR}

In [ ]:
!git clone https://github.com/YOUR_USER/roadscan.git {DATA_DIR / 'repo'}
!pip install -q ultralytics imagehash scikit-learn tqdm pyyaml

In [ ]:
import sys
sys.path.insert(0, str(DATA_DIR / 'repo' / 'src'))
from data.convert import convert_dataset
YOLO_DIR = DATA_DIR / 'yolo'
stats = convert_dataset(str(RAW_DIR), str(YOLO_DIR), copy_images=True)
print(f'Converted {stats["total_images"]} images')

In [ ]:
from data.group import group_images, save_groups
image_paths = sorted([str(f) for f in (YOLO_DIR / 'images').rglob('*.jpg')])
groups = group_images(image_paths, window=8, max_distance=10, assume_sequential=True)
save_groups(image_paths, groups, str(DATA_DIR / 'groups.json'))
print(f'Grouped into {max(groups)+1} groups')

In [ ]:
from data.split import create_splits
split_stats = create_splits(str(YOLO_DIR), str(DATA_DIR / 'groups.json'), str(DATA_DIR / 'splits'), test_size=0.1, val_size=0.1)
print(split_stats)

In [ ]:
yaml_content = f'''path: {YOLO_DIR}
train: splits/train.txt
val: splits/val.txt
test: splits/test.txt
nc: 4
names:
  0: D00
  1: D10
  2: D20
  3: D40
'''
with open(DATA_DIR / 'data.yaml', 'w') as f:
    f.write(yaml_content)
print('Ready for training!')